In [8]:
!pip uninstall -y -q bitsandbytes 2>/dev/null
!rm -rf /usr/local/lib/python3.12/dist-packages/bitsandbytes* 2>/dev/null
!pip install -q transformers==4.57.1 accelerate==1.10.1 peft==0.17.1 scipy scikit-learn matplotlib
print("Install complete")

Install complete


In [9]:
import os, sys, json, pickle, time, itertools, warnings, random, string, re, gc, types
import importlib.util
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ---- bitsandbytes stub ----
def _dummy_spec(name):
    return importlib.util.spec_from_loader(name, loader=None)

class _StubClass:
    def __init__(self, *a, **kw): pass
    def __call__(self, *a, **kw): return None

_bnb_stub = types.ModuleType("bitsandbytes")
_bnb_stub.__version__ = "0.0.0-stub"; _bnb_stub.__path__ = []
_bnb_stub.__spec__ = _dummy_spec("bitsandbytes")

_bnb_nn = types.ModuleType("bitsandbytes.nn")
_bnb_nn.Linear8bitLt = _StubClass; _bnb_nn.Linear4bit = _StubClass
_bnb_nn.Int8Params = _StubClass; _bnb_nn.Params4bit = _StubClass
_bnb_nn.__spec__ = _dummy_spec("bitsandbytes.nn"); _bnb_stub.nn = _bnb_nn

_bnb_optim = types.ModuleType("bitsandbytes.optim")
_bnb_optim.Adam8bit = _StubClass; _bnb_optim.AdamW8bit = _StubClass
_bnb_optim.Adam32bit = _StubClass
_bnb_optim.__spec__ = _dummy_spec("bitsandbytes.optim"); _bnb_stub.optim = _bnb_optim

_bnb_func = types.ModuleType("bitsandbytes.functional")
_bnb_func.__spec__ = _dummy_spec("bitsandbytes.functional"); _bnb_stub.functional = _bnb_func

_bnb_triton = types.ModuleType("bitsandbytes.triton")
_bnb_triton.__spec__ = _dummy_spec("bitsandbytes.triton"); _bnb_stub.triton = _bnb_triton

sys.modules["bitsandbytes"] = _bnb_stub
sys.modules["bitsandbytes.nn"] = _bnb_nn
sys.modules["bitsandbytes.optim"] = _bnb_optim
sys.modules["bitsandbytes.functional"] = _bnb_func
sys.modules["bitsandbytes.triton"] = _bnb_triton

assert importlib.util.find_spec("bitsandbytes") is not None
print(f"bitsandbytes stub: v{_bnb_stub.__version__}")

import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
from collections import Counter, deque
import matplotlib.pyplot as plt
from datetime import timedelta

from transformers import (
    Qwen2_5_VLForConditionalGeneration, AutoProcessor,
    Owlv2Processor, Owlv2ForObjectDetection,
    CLIPModel, CLIPProcessor,
)
from peft import LoraConfig, get_peft_model
warnings.filterwarnings("ignore")

TRAIN_ANN     = "/kaggle/input/datasets/lhanhsin/vizwiz/Annotations/Annotations/train.json"
VAL_ANN       = "/kaggle/input/datasets/lhanhsin/vizwiz/Annotations/Annotations/val.json"
TRAIN_IMG_DIR = "/kaggle/input/datasets/lhanhsin/vizwiz/train/train"
VAL_IMG_DIR   = "/kaggle/input/datasets/lhanhsin/vizwiz/val/val"

QWEN_MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
OWL_MODEL_ID  = "google/owlv2-base-patch16-ensemble"
CLIP_MODEL_ID = "openai/clip-vit-base-patch32"

IMAGE_SIZE = 336
PIXEL_BUDGET = IMAGE_SIZE * IMAGE_SIZE

SYSTEM_PROMPT = (
    "You are a careful visual assistant helping a blind user. "
    "Answer using only visible information. "
    "If the answer cannot be determined, respond exactly: unanswerable."
)

CACHE_DIR = "/kaggle/working/cache"
LORA_DIR  = "/kaggle/working/qwen_lora_adapter"
os.makedirs(CACHE_DIR, exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ---------- Sample sizes ----------
N_FINETUNE = 1800
N_RISK_TRAIN = 100     # risk module training
N_POLICY_TRAIN = 100   # policy training
N_CALIBRATION = 500    # conformal calibration set
N_TEST = 400           # test set (kept small for conformal guarantee to hold)
FT_EPOCHS = 3

print("Config ready")

bitsandbytes stub: v0.0.0-stub
Config ready


In [10]:
def load_vizwiz(ann_path, img_dir):
    with open(ann_path) as f:
        data = json.load(f)
    return [{"image_id": e["image"],
             "image_path": os.path.join(img_dir, e["image"]),
             "question": e["question"],
             "answers": [a["answer"] for a in e["answers"]]}
            for e in data if os.path.exists(os.path.join(img_dir, e["image"]))]

def answer_type_bucket(sample):
    ans = [a.lower().strip() for a in sample["answers"]]
    if any(a in ("unanswerable", "unsuitable", "") for a in ans): return "unanswerable"
    if any(a in ("yes", "no") for a in ans): return "yesno"
    if any(a.isdigit() for a in ans): return "number"
    return "other"

all_train_pool = load_vizwiz(TRAIN_ANN, TRAIN_IMG_DIR)
val_pool       = load_vizwiz(VAL_ANN,  VAL_IMG_DIR)

rng = random.Random(SEED)
rng.shuffle(all_train_pool)

finetune_samples   = all_train_pool[:N_FINETUNE]
risk_train_samples = all_train_pool[N_FINETUNE:N_FINETUNE + N_RISK_TRAIN]
policy_train_samples = all_train_pool[N_FINETUNE + N_RISK_TRAIN:
                                      N_FINETUNE + N_RISK_TRAIN + N_POLICY_TRAIN]

# Calibration + test sets (stratified from validation pool)
buckets = {}
for s in val_pool:
    buckets.setdefault(answer_type_bucket(s), []).append(s)

full_dist = {k: len(v) / len(val_pool) for k, v in buckets.items()}
val_selected = []
for k, v in buckets.items():
    rng.shuffle(v)
    take = round(full_dist[k] * (N_CALIBRATION + N_TEST))
    val_selected.extend(v[:take])

rng.shuffle(val_selected)
calibration_samples = val_selected[:N_CALIBRATION]
test_samples        = val_selected[N_CALIBRATION:N_CALIBRATION + N_TEST]

# Verify disjointness
def ids(s): return set(x["image_id"] for x in s)
assert ids(finetune_samples)     & ids(risk_train_samples)   == set()
assert ids(finetune_samples)     & ids(policy_train_samples) == set()
assert ids(finetune_samples)     & ids(test_samples)         == set()
assert ids(risk_train_samples)   & ids(policy_train_samples) == set()
assert ids(risk_train_samples)   & ids(calibration_samples)  == set()
assert ids(policy_train_samples) & ids(calibration_samples)  == set()
assert ids(calibration_samples)  & ids(test_samples)         == set()

print(f"Fine-tune: {len(finetune_samples)}")
print(f"Risk-train: {len(risk_train_samples)}")
print(f"Policy-train: {len(policy_train_samples)}")
print(f"Calibration: {len(calibration_samples)}")
print(f"Test: {len(test_samples)}")

Fine-tune: 1800
Risk-train: 100
Policy-train: 100
Calibration: 500
Test: 400


In [22]:
ARTICLES = {"a", "an", "the"}
PUNCT = str.maketrans("", "", string.punctuation)

def normalize_answer(ans):
    ans = str(ans).lower().strip().translate(PUNCT)
    return " ".join(w for w in ans.split() if w not in ARTICLES)

def vqa_accuracy(candidate, answers):
    """Standard VQA accuracy: for each ground-truth answer, count how many of the
    other 9 annotators gave the same answer, cap at 1.0, average across all 10."""
    cand = normalize_answer(candidate)
    gt = [normalize_answer(a) for a in answers]
    if len(gt) < 2:
        return float(cand in gt)
    scores = []
    for i in range(len(gt)):
        others = gt[:i] + gt[i+1:]
        # FIX: was `a == c`, must be `a == cand`
        scores.append(min(sum(gt_ans == cand for gt_ans in others) / 3, 1))
    return float(np.mean(scores))

def majority_answer(answers):
    gt = [normalize_answer(a) for a in answers]
    return Counter(gt).most_common(1)[0][0]

print("Metrics ready")

Metrics ready


In [12]:
gc.collect(); torch.cuda.empty_cache()
torch.set_default_dtype(torch.float16)

qwen_processor = AutoProcessor.from_pretrained(
    QWEN_MODEL_ID,
    size={"height": IMAGE_SIZE, "width": IMAGE_SIZE},
    min_pixels=PIXEL_BUDGET, max_pixels=PIXEL_BUDGET,
    do_resize=True, resample=Image.BICUBIC,
)
qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    QWEN_MODEL_ID, dtype=torch.float16, device_map="auto",
    low_cpu_mem_usage=True, max_memory={0: "12GiB", 1: "12GiB", "cpu": "20GiB"},
)
qwen_model.eval()
for p in qwen_model.parameters(): p.requires_grad = False
print("Qwen loaded")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Qwen loaded


In [14]:
qwen_model.gradient_checkpointing_enable()
qwen_model.enable_input_require_grads()

lora_config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, bias="none",
                          target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM")
qwen_model = get_peft_model(qwen_model, lora_config)
qwen_model.print_trainable_parameters()

def prepare_image(image):
    return image.convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BICUBIC)

def process_vision_info(messages):
    images, videos = [], []
    for m in messages:
        if m["role"] == "user":
            for c in m["content"]:
                if c["type"] == "image": images.append(c["image"])
                elif c["type"] == "video": videos.append(c["video"])
    return images, videos

def input_device():
    try: return next(qwen_model.parameters()).device
    except: return torch.device("cuda:0")

def build_training_example(sample):
    img = prepare_image(Image.open(sample["image_path"]))
    target = majority_answer(sample["answers"])
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": sample["question"]},
        ]},
    ]
    prompt = qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    images, videos = process_vision_info(messages)
    enc = qwen_processor(text=[prompt], images=images, videos=videos or None, return_tensors="pt")
    # EOS appended — prevents runaway generation
    target_with_eos = target + qwen_processor.tokenizer.eos_token
    full = qwen_processor(text=[prompt + target_with_eos],
                           images=images, videos=videos or None, return_tensors="pt")
    labels = full["input_ids"].clone()
    labels[:, :enc["input_ids"].shape[1]] = -100
    full["labels"] = labels
    return full

BATCH_SIZE, LR = 4, 2e-4
optimizer = torch.optim.AdamW((p for p in qwen_model.parameters() if p.requires_grad), lr=LR)
qwen_model.train()
skipped = 0

for epoch in range(FT_EPOCHS):
    rng.shuffle(finetune_samples)
    accum, running_loss = 0, 0
    optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(finetune_samples, desc=f"LoRA epoch {epoch+1}/{FT_EPOCHS}")
    for sample in pbar:
        try:
            enc = build_training_example(sample)
            dev = input_device()
            enc = {k: v.to(dev) if torch.is_tensor(v) else v for k, v in enc.items()}
            out = qwen_model(**enc)
            loss = out.loss / BATCH_SIZE
            if not torch.isfinite(loss):
                del enc, out, loss; optimizer.zero_grad(set_to_none=True); accum = 0
                torch.cuda.empty_cache(); continue
            loss.backward()
            running_loss += loss.item(); accum += 1
            if accum == BATCH_SIZE:
                torch.nn.utils.clip_grad_norm_(
                    (p for p in qwen_model.parameters() if p.requires_grad), 0.3)
                optimizer.step(); optimizer.zero_grad(set_to_none=True)
                accum = 0; pbar.set_postfix(loss=f"{running_loss:.4f}"); running_loss = 0
            del enc, out, loss; torch.cuda.empty_cache()
        except torch.cuda.OutOfMemoryError:
            skipped += 1; optimizer.zero_grad(set_to_none=True); accum = 0
            gc.collect(); torch.cuda.empty_cache()
        except Exception:
            skipped += 1; optimizer.zero_grad(set_to_none=True); accum = 0
            torch.cuda.empty_cache()

qwen_model.eval()
qwen_model.save_pretrained(LORA_DIR); qwen_processor.save_pretrained(LORA_DIR)
print(f"LoRA complete. Skipped: {skipped}")

trainable params: 1,843,200 || all params: 3,756,466,176 || trainable%: 0.0491


LoRA epoch 1/3:   0%|          | 0/1800 [00:00<?, ?it/s]

LoRA epoch 2/3:   0%|          | 0/1800 [00:00<?, ?it/s]

LoRA epoch 3/3:   0%|          | 0/1800 [00:00<?, ?it/s]

LoRA complete. Skipped: 0


In [26]:
!rm -f /kaggle/working/cache/*.pkl
!ls -la /kaggle/working/cache/ 2>/dev/null || echo "(cache dir empty)"

total 8
drwxr-xr-x 2 root root 4096 Sep 27 16:42 .
drwxr-xr-x 5 root root 4096 Sep 27 16:12 ..


In [27]:
# Confirm models and functions are still in memory from earlier cells
import numpy as np, torch
from PIL import Image
from tqdm.auto import tqdm
from collections import Counter
import pickle, os, json, re, time

# Check that the key names are defined
required = ["qwen_model", "qwen_processor", "owl_model", "owl_processor",
            "clip_model", "clip_processor",
            "qwen_answer", "owl_features", "clip_features",
            "qwen_verbalized_confidence", "vqa_accuracy", "safety_proxy_label",
            "risk_train_samples", "policy_train_samples",
            "calibration_samples", "test_samples",
            "CACHE_DIR", "IMAGE_SIZE"]

missing = [n for n in required if n not in dir()]
if missing:
    print(f"❌ Missing from namespace: {missing}")
    print("   Re-run earlier cells before proceeding.")
else:
    print("✅ All prerequisites loaded")
    print(f"   CACHE_DIR = {CACHE_DIR}")
    print(f"   risk_train: {len(risk_train_samples)}")
    print(f"   policy_train: {len(policy_train_samples)}")
    print(f"   calibration: {len(calibration_samples)}")
    print(f"   test: {len(test_samples)}")

✅ All prerequisites loaded
   CACHE_DIR = /kaggle/working/cache
   risk_train: 100
   policy_train: 100
   calibration: 500
   test: 400


In [28]:
!rm -f /kaggle/working/cache/*.pkl
!ls -la /kaggle/working/cache/

total 8
drwxr-xr-x 2 root root 4096 Sep 27 16:42 .
drwxr-xr-x 5 root root 4096 Sep 27 16:12 ..


In [29]:
risk_train_raw   = build_cache(risk_train_samples,   "risk_train",   f"{CACHE_DIR}/risk_train_raw_cache.pkl")
policy_train_raw = build_cache(policy_train_samples, "policy_train", f"{CACHE_DIR}/policy_train_raw_cache.pkl")
calibration_raw  = build_cache(calibration_samples,  "calibration",  f"{CACHE_DIR}/calibration_raw_cache.pkl")
test_raw         = build_cache(test_samples,         "test",         f"{CACHE_DIR}/test_raw_cache.pkl")

[risk_train]:   0%|          | 0/100 [00:00<?, ?it/s]

risk_train: 100/100 cached


[policy_train]:   0%|          | 0/100 [00:00<?, ?it/s]

policy_train: 100/100 cached


[calibration]:   0%|          | 0/500 [00:00<?, ?it/s]

    owl failed: The size of tensor a (18) must match the size of tensor b (1
calibration: 500/500 cached


[test]:   0%|          | 0/400 [00:00<?, ?it/s]

test: 400/400 cached


In [30]:
def summarize_cache(name, cache):
    if not cache:
        print(f"❌ {name}: EMPTY")
        return
    accs  = [e["base_accuracy"] for e in cache]
    vcs   = [e["verbalized_confidence"] for e in cache]
    owls  = [e["owl_conf"] for e in cache]
    clips = [e["clip_sim"] for e in cache]
    pms   = [e["state_raw"][4] for e in cache]   # prob_mean
    pvs   = [e["state_raw"][5] for e in cache]   # prob_var
    pmin  = [e["state_raw"][6] for e in cache]   # prob_min

    print(f"✅ {name}: {len(cache)} samples")
    print(f"   base_acc        min={min(accs):.3f}  max={max(accs):.3f}  mean={np.mean(accs):.3f}  std={np.std(accs):.3f}")
    print(f"   verbalized_conf min={min(vcs):.3f}  max={max(vcs):.3f}  mean={np.mean(vcs):.3f}  std={np.std(vcs):.3f}")
    print(f"   owl_conf        min={min(owls):.3f}  max={max(owls):.3f}  mean={np.mean(owls):.3f}  std={np.std(owls):.3f}")
    print(f"   clip_sim        min={min(clips):.3f}  max={max(clips):.3f}  mean={np.mean(clips):.3f}  std={np.std(clips):.3f}")
    print(f"   prob_mean       min={min(pms):.3f}  max={max(pms):.3f}  mean={np.mean(pms):.3f}  std={np.std(pms):.3f}")
    print(f"   prob_var        min={min(pvs):.3f}  max={max(pvs):.3f}  mean={np.mean(pvs):.3f}  std={np.std(pvs):.3f}")
    print(f"   prob_min        min={min(pmin):.3f}  max={max(pmin):.3f}  mean={np.mean(pmin):.3f}  std={np.std(pmin):.3f}")

    # Check that proxy labels aren't degenerate
    labels = [e["unsafe_proxy_label"] for e in cache]
    print(f"   proxy labels    0={labels.count(0)}  1={labels.count(1)}")
    print()

summarize_cache("risk_train",   risk_train_raw)
summarize_cache("policy_train", policy_train_raw)
summarize_cache("calibration",  calibration_raw)
summarize_cache("test",         test_raw)

✅ risk_train: 100 samples
   base_acc        min=0.000  max=1.000  mean=0.645  std=0.432
   verbalized_conf min=0.000  max=1.000  mean=0.533  std=0.407
   owl_conf        min=0.000  max=0.772  mean=0.165  std=0.223
   clip_sim        min=0.000  max=0.362  mean=0.241  std=0.092
   prob_mean       min=0.355  max=1.000  mean=0.854  std=0.125
   prob_var        min=0.000  max=0.130  mean=0.034  std=0.036
   prob_min        min=0.078  max=1.000  mean=0.651  std=0.248
   proxy labels    0=99  1=1

✅ policy_train: 100 samples
   base_acc        min=0.000  max=1.000  mean=0.634  std=0.412
   verbalized_conf min=0.000  max=1.000  mean=0.574  std=0.410
   owl_conf        min=0.000  max=0.957  mean=0.187  std=0.268
   clip_sim        min=0.000  max=0.340  mean=0.231  std=0.094
   prob_mean       min=0.465  max=1.000  mean=0.872  std=0.112
   prob_var        min=0.000  max=0.135  mean=0.031  std=0.036
   prob_min        min=0.144  max=0.999  mean=0.678  std=0.256
   proxy labels    0=100  1=0

✅ c

In [31]:
def safety_proxy_label_v2(candidate_answer, base_accuracy):
    """Unsafe = confidently wrong. The model committed to an answer that is
    factually incorrect, rather than hedging with 'unanswerable'/'unsuitable'.
    This captures the safety concern directly: a wrong answer the model
    presents without qualification can actively mislead a blind user."""
    if base_accuracy >= 0.5:
        return 0   # correct
    ans = str(candidate_answer).strip().lower()
    if ans in ("unanswerable", "unsuitable", "") or ans.startswith("unanswerable") or ans.startswith("unsuitable"):
        return 0   # hedged, safe
    return 1       # confidently wrong

# Relabel every entry in every cache in place
for split_name, cache in [("risk_train", risk_train_raw),
                           ("policy_train", policy_train_raw),
                           ("calibration", calibration_raw),
                           ("test", test_raw)]:
    for e in cache:
        e["unsafe_proxy_label"] = safety_proxy_label_v2(e["candidate_answer"],
                                                        e["base_accuracy"])

    pos = sum(e["unsafe_proxy_label"] for e in cache)
    print(f"{split_name:16s}: {pos:4d}/{len(cache):4d} positives ({100*pos/len(cache):.1f}%)")

# Re-save all four caches
for split_name, cache in [("risk_train", risk_train_raw),
                           ("policy_train", policy_train_raw),
                           ("calibration", calibration_raw),
                           ("test", test_raw)]:
    with open(f"{CACHE_DIR}/{split_name}_raw_cache.pkl", "wb") as f:
        pickle.dump(cache, f)
print("\n✅ All caches relabeled and saved")

risk_train      :   27/ 100 positives (27.0%)
policy_train    :   25/ 100 positives (25.0%)
calibration     :  128/ 500 positives (25.6%)
test            :  118/ 400 positives (29.5%)

✅ All caches relabeled and saved


In [32]:
def summarize_labels(name, cache):
    labels = [e["unsafe_proxy_label"] for e in cache]
    n = len(labels)
    pos = sum(labels)
    neg = n - pos
    print(f"{name:16s}: total={n}  pos={pos} ({100*pos/n:.1f}%)  neg={neg} ({100*neg/n:.1f}%)")

summarize_labels("risk_train",   risk_train_raw)
summarize_labels("policy_train", policy_train_raw)
summarize_labels("calibration",  calibration_raw)
summarize_labels("test",         test_raw)

risk_train      : total=100  pos=27 (27.0%)  neg=73 (73.0%)
policy_train    : total=100  pos=25 (25.0%)  neg=75 (75.0%)
calibration     : total=500  pos=128 (25.6%)  neg=372 (74.4%)
test            : total=400  pos=118 (29.5%)  neg=282 (70.5%)


In [33]:
#sanity check 
def show_examples(cache, k=5):
    positives = [e for e in cache if e["unsafe_proxy_label"] == 1]
    print(f"{len(positives)} positives in this split. First {min(k, len(positives))}:")
    for e in positives[:k]:
        print(f"  Q: {e['question'][:70]!r}")
        print(f"     A: {e['candidate_answer']!r}  (acc={e['base_accuracy']:.2f})")
        print(f"     GT: {e['answers'][:3]}")
        print()

show_examples(test_raw, k=5)

118 positives in this split. First 5:
  Q: 'What is this item?'
     A: 'book'  (acc=0.00)
     GT: ['advertisement', 'box', 'invitation']

  Q: 'What is in this can?'
     A: 'beer'  (acc=0.00)
     GT: ['pop', 'unsuitable', 'unsuitable']

  Q: 'On the left.'
     A: 'girl'  (acc=0.00)
     GT: ['unanswerable', 'red picture frame', 'unanswerable']

  Q: 'What do you see?'
     A: 'ceiling'  (acc=0.00)
     GT: ['back car seat', 'back car seat', 'headrest']

  Q: 'Can you please read this menu to me.'
     A: 'sushi bar appetizers\nsashimi\nmaguro tuna\nmaguro shrimp\ntako'  (acc=0.00)
     GT: ['unanswerable', 'unanswerable', 'no too much text too far away']



In [34]:
!rm -rf /kaggle/working/aerpn_export
!mkdir -p /kaggle/working/aerpn_export

!cp /kaggle/working/cache/risk_train_raw_cache.pkl   /kaggle/working/aerpn_export/
!cp /kaggle/working/cache/policy_train_raw_cache.pkl /kaggle/working/aerpn_export/
!cp /kaggle/working/cache/calibration_raw_cache.pkl  /kaggle/working/aerpn_export/
!cp /kaggle/working/cache/test_raw_cache.pkl         /kaggle/working/aerpn_export/
!cp -r /kaggle/working/qwen_lora_adapter             /kaggle/working/aerpn_export/

meta = {
    "n_risk_train":   len(risk_train_raw),
    "n_policy_train": len(policy_train_raw),
    "n_calibration":  len(calibration_raw),
    "n_test":         len(test_raw),
    "image_size":     IMAGE_SIZE,
    "ft_epochs":      FT_EPOCHS,
    "n_finetune":     N_FINETUNE,
    "proxy_definition": "confidently_wrong_v2",
}
with open("/kaggle/working/aerpn_export/metadata.json", "w") as f:
    json.dump(meta, f, indent=2)

!ls -lh /kaggle/working/aerpn_export/

total 392K
-rw-r--r-- 1 root root 172K Sep 27 17:10 calibration_raw_cache.pkl
-rw-r--r-- 1 root root  198 Sep 27 17:10 metadata.json
-rw-r--r-- 1 root root  33K Sep 27 17:10 policy_train_raw_cache.pkl
drwxr-xr-x 2 root root 4.0K Sep 27 17:10 qwen_lora_adapter
-rw-r--r-- 1 root root  34K Sep 27 17:10 risk_train_raw_cache.pkl
-rw-r--r-- 1 root root 137K Sep 27 17:10 test_raw_cache.pkl


In [13]:
owl_processor = Owlv2Processor.from_pretrained(OWL_MODEL_ID)
owl_model = Owlv2ForObjectDetection.from_pretrained(OWL_MODEL_ID).to("cuda:0").eval()
for p in owl_model.parameters(): p.requires_grad = False

clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)
clip_model = CLIPModel.from_pretrained(CLIP_MODEL_ID).to("cuda:0").eval()
for p in clip_model.parameters(): p.requires_grad = False

print("OWLv2 + CLIP loaded")

OWLv2 + CLIP loaded


In [15]:
def qwen_answer(image, question, max_new_tokens=20):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": question},
        ]},
    ]
    text = qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    images, videos = process_vision_info(messages)
    dev = input_device()
    inputs = qwen_processor(text=[text], images=images, videos=videos or None,
                            return_tensors="pt").to(dev)
    with torch.no_grad():
        outputs = qwen_model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            return_dict_in_generate=True, output_scores=True,
            do_sample=False, use_cache=True,
            pad_token_id=qwen_processor.tokenizer.pad_token_id,
            eos_token_id=qwen_processor.tokenizer.eos_token_id,
        )
    input_len = inputs.input_ids.shape[1]
    answer = qwen_processor.decode(outputs.sequences[0, input_len:], skip_special_tokens=True).strip()
    token_probs = [torch.softmax(s, dim=-1).max().item() for s in outputs.scores] if outputs.scores else [0.5]
    del inputs, outputs; torch.cuda.empty_cache()
    return answer, float(np.mean(token_probs)), float(np.var(token_probs)), float(np.min(token_probs))

def qwen_verbalized_confidence(image, question, answer, max_new_tokens=10):
    """Ask Qwen to rate its confidence in its own answer, 0-100."""
    prompt = (f"Question: {question}\nYour answer: {answer}\n"
              "On a scale of 0 to 100, how confident are you that this answer is correct? "
              "Output only the number.")
    messages = [
        {"role": "system", "content": "You are a careful assistant."},
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt},
        ]},
    ]
    text = qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    images, videos = process_vision_info(messages)
    dev = input_device()
    inputs = qwen_processor(text=[text], images=images, videos=videos or None,
                            return_tensors="pt").to(dev)
    with torch.no_grad():
        outputs = qwen_model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True,
            pad_token_id=qwen_processor.tokenizer.pad_token_id,
            eos_token_id=qwen_processor.tokenizer.eos_token_id,
        )
    out = qwen_processor.decode(outputs[0, inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    m = re.search(r"(\d+)", out)
    del inputs, outputs
    return float(m.group(1)) / 100.0 if m else 0.5

def owl_features(image, query_text, threshold=0.1):
    if not query_text or query_text.lower().startswith("unanswerable"):
        return 0.0, 0
    try:
        query_text = query_text.strip()[:200]
        inputs = owl_processor(images=image, text=[query_text], return_tensors="pt").to("cuda:0")
        with torch.no_grad(): outputs = owl_model(**inputs)
        ts = torch.tensor([image.size[::-1]]).to("cuda:0")
        res = owl_processor.post_process_object_detection(outputs, threshold=threshold, target_sizes=ts)
        scores = res[0]["scores"].cpu().numpy()
        del inputs, outputs
        return (float(np.max(scores)), int(np.sum(scores < 0.3))) if len(scores) else (0.0, 0)
    except Exception as e:
        print(f"    owl failed: {str(e)[:60]}"); return 0.0, 0

def clip_features(image, answer_text, question_text):
    if not answer_text or answer_text.lower().startswith("unanswerable"):
        return 0.0, 0.0
    try:
        answer_text = answer_text.replace("\n", " ").strip()[:100]
        question_text = question_text.replace("\n", " ").strip()[:100]
        words = answer_text.split()
        fragments = [" ".join(words[:i+1]) for i in range(len(words))] or [answer_text]
        texts = [f"{question_text} {f}"[:200] for f in fragments]
        inputs = clip_processor(text=texts, images=image, return_tensors="pt",
                                 padding=True, truncation=True, max_length=77).to("cuda:0")
        with torch.no_grad():
            out = clip_model(**inputs)
            img_e = out.image_embeds / out.image_embeds.norm(dim=-1, keepdim=True)
            txt_e = out.text_embeds / out.text_embeds.norm(dim=-1, keepdim=True)
            sims = (img_e @ txt_e.T).squeeze(0).cpu().numpy()
        del inputs, out
        return float(sims[-1]), float(np.var(sims))
    except Exception as e:
        print(f"    clip failed: {str(e)[:60]}"); return 0.0, 0.0

# Proxy label — independent of model
SAFETY_KEYWORDS = {
    "medication": ["medicine", "pill", "medication", "dosage", "prescription", "aspirin"],
    "traffic":    ["traffic", "car", "crosswalk", "signal", "pedestrian", "stop sign"],
    "hazard":     ["warning", "danger", "hazard", "poison", "toxic", "fire", "sharp", "burn"],
    "expiration": ["expire", "expiration", "spoiled", "moldy", "rotten"],
    "allergen":   ["allergen", "peanut", "gluten", "dairy", "contains"],
    "medical":    ["hospital", "doctor", "symptom", "injury", "blood"],
}
SAFETY_FLAT = [kw for kws in SAFETY_KEYWORDS.values() for kw in kws]

def safety_proxy_label(question, base_accuracy):
    if base_accuracy >= 0.5: return 0
    return int(any(kw in question.lower() for kw in SAFETY_FLAT))

print("Feature functions ready")

Feature functions ready


In [16]:
def build_cache(samples, split_name, cache_file):
    if os.path.exists(cache_file):
        with open(cache_file, 'rb') as f: return pickle.load(f)

    cache, error_log = [], []
    pbar = tqdm(samples, desc=f"[{split_name}]")
    for idx, sample in enumerate(pbar):
        try:
            img = Image.open(sample["image_path"]).convert("RGB")
            q, answers = sample["question"], sample["answers"]

            answer, prob_mean, prob_var, prob_min = qwen_answer(img, q)
            owl_conf, owl_low = owl_features(img, answer)
            clip_sim, clip_var = clip_features(img, answer, q)
            verbalized_conf = qwen_verbalized_confidence(img, q, answer)
            acc = vqa_accuracy(answer, answers)

            state = np.array([owl_conf, owl_low, clip_sim, clip_var,
                               prob_mean, prob_var, prob_min], dtype=np.float32)

            cache.append({
                "image_id": sample["image_id"], "question": q, "answers": answers,
                "candidate_answer": answer, "base_accuracy": acc,
                "state_raw": state,
                "verbalized_confidence": verbalized_conf,
                "unsafe_proxy_label": safety_proxy_label(q, acc),
                "owl_conf": owl_conf, "clip_sim": clip_sim,
            })
        except Exception as e:
            error_log.append((idx, sample.get("image_id"), type(e).__name__, str(e)[:100]))
            continue
        if len(cache) % 50 == 0:
            torch.cuda.empty_cache()
            pbar.set_postfix({"ok": len(cache), "err": len(error_log)})

    if error_log:
        print(f"\n{len(error_log)}/{len(samples)} failed in {split_name}:")
        for e in error_log[:5]: print(f"  {e}")

    with open(cache_file, "wb") as f: pickle.dump(cache, f)
    print(f"{split_name}: {len(cache)}/{len(samples)} cached")
    return cache

!rm -f /kaggle/working/cache/*.pkl

risk_train_raw   = build_cache(risk_train_samples,   "risk_train",   f"{CACHE_DIR}/risk_train_raw_cache.pkl")
policy_train_raw = build_cache(policy_train_samples, "policy_train", f"{CACHE_DIR}/policy_train_raw_cache.pkl")
calibration_raw  = build_cache(calibration_samples,  "calibration",  f"{CACHE_DIR}/calibration_raw_cache.pkl")
test_raw         = build_cache(test_samples,         "test",         f"{CACHE_DIR}/test_raw_cache.pkl")

[risk_train]:   0%|          | 0/100 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



100/100 failed in risk_train:
  (0, 'VizWiz_train_00014435.jpg', 'NameError', "name 'a' is not defined")
  (1, 'VizWiz_train_00002063.jpg', 'NameError', "name 'a' is not defined")
  (2, 'VizWiz_train_00003906.jpg', 'NameError', "name 'a' is not defined")
  (3, 'VizWiz_train_00007237.jpg', 'NameError', "name 'a' is not defined")
  (4, 'VizWiz_train_00002396.jpg', 'NameError', "name 'a' is not defined")
risk_train: 0/100 cached


[policy_train]:   0%|          | 0/100 [00:00<?, ?it/s]


100/100 failed in policy_train:
  (0, 'VizWiz_train_00000921.jpg', 'NameError', "name 'a' is not defined")
  (1, 'VizWiz_train_00015167.jpg', 'NameError', "name 'a' is not defined")
  (2, 'VizWiz_train_00014251.jpg', 'NameError', "name 'a' is not defined")
  (3, 'VizWiz_train_00012856.jpg', 'NameError', "name 'a' is not defined")
  (4, 'VizWiz_train_00014572.jpg', 'NameError', "name 'a' is not defined")
policy_train: 0/100 cached


[calibration]:   0%|          | 0/500 [00:00<?, ?it/s]

    owl failed: The size of tensor a (18) must match the size of tensor b (1

500/500 failed in calibration:
  (0, 'VizWiz_val_00003083.jpg', 'NameError', "name 'a' is not defined")
  (1, 'VizWiz_val_00003518.jpg', 'NameError', "name 'a' is not defined")
  (2, 'VizWiz_val_00002811.jpg', 'NameError', "name 'a' is not defined")
  (3, 'VizWiz_val_00001542.jpg', 'NameError', "name 'a' is not defined")
  (4, 'VizWiz_val_00003441.jpg', 'NameError', "name 'a' is not defined")
calibration: 0/500 cached


[test]:   0%|          | 0/400 [00:00<?, ?it/s]


400/400 failed in test:
  (0, 'VizWiz_val_00001423.jpg', 'NameError', "name 'a' is not defined")
  (1, 'VizWiz_val_00003006.jpg', 'NameError', "name 'a' is not defined")
  (2, 'VizWiz_val_00002072.jpg', 'NameError', "name 'a' is not defined")
  (3, 'VizWiz_val_00000267.jpg', 'NameError', "name 'a' is not defined")
  (4, 'VizWiz_val_00003278.jpg', 'NameError', "name 'a' is not defined")
test: 0/400 cached


In [19]:
def qwen_verbalized_confidence(image, question, answer, max_new_tokens=10):
    """Ask Qwen to rate its confidence in its own answer, 0-100."""
    # Fold the system instruction into the user prompt — avoids system-role
    # incompatibility with Qwen2.5-VL's chat template.
    prompt = (
        "You are a careful assistant. "
        f"Question: {question}\n"
        f"Your answer: {answer}\n"
        "On a scale of 0 to 100, how confident are you that this answer is correct? "
        "Output only the number."
    )
    messages = [
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt},
        ]},
    ]
    text = qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    images, videos = process_vision_info(messages)
    dev = input_device()
    inputs = qwen_processor(text=[text], images=images, videos=videos or None,
                            return_tensors="pt").to(dev)
    with torch.no_grad():
        outputs = qwen_model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True,
            pad_token_id=qwen_processor.tokenizer.pad_token_id,
            eos_token_id=qwen_processor.tokenizer.eos_token_id,
        )
    out = qwen_processor.decode(outputs[0, inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    m = re.search(r"(\d+)", out)
    del inputs, outputs
    return float(m.group(1)) / 100.0 if m else 0.5

In [20]:
def qwen_answer(image, question, max_new_tokens=20):
    # Fold SYSTEM_PROMPT into the user message — avoids system-role issue
    prompt_text = f"{SYSTEM_PROMPT}\n\n{question}"
    messages = [
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt_text},
        ]},
    ]
    text = qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    images, videos = process_vision_info(messages)
    dev = input_device()
    inputs = qwen_processor(text=[text], images=images, videos=videos or None,
                            return_tensors="pt").to(dev)
    with torch.no_grad():
        outputs = qwen_model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            return_dict_in_generate=True, output_scores=True,
            do_sample=False, use_cache=True,
            pad_token_id=qwen_processor.tokenizer.pad_token_id,
            eos_token_id=qwen_processor.tokenizer.eos_token_id,
        )
    input_len = inputs.input_ids.shape[1]
    answer = qwen_processor.decode(outputs.sequences[0, input_len:], skip_special_tokens=True).strip()
    token_probs = [torch.softmax(s, dim=-1).max().item() for s in outputs.scores] if outputs.scores else [0.5]
    del inputs, outputs; torch.cuda.empty_cache()
    return answer, float(np.mean(token_probs)), float(np.var(token_probs)), float(np.min(token_probs))

In [35]:
!ls -lh /kaggle/working/aerpn_export/
!cat /kaggle/working/aerpn_export/metadata.json

total 392K
-rw-r--r-- 1 root root 172K Sep 27 17:10 calibration_raw_cache.pkl
-rw-r--r-- 1 root root  198 Sep 27 17:10 metadata.json
-rw-r--r-- 1 root root  33K Sep 27 17:10 policy_train_raw_cache.pkl
drwxr-xr-x 2 root root 4.0K Sep 27 17:10 qwen_lora_adapter
-rw-r--r-- 1 root root  34K Sep 27 17:10 risk_train_raw_cache.pkl
-rw-r--r-- 1 root root 137K Sep 27 17:10 test_raw_cache.pkl
{
  "n_risk_train": 100,
  "n_policy_train": 100,
  "n_calibration": 500,
  "n_test": 400,
  "image_size": 336,
  "ft_epochs": 3,
  "n_finetune": 1800,
  "proxy_definition": "confidently_wrong_v2"
}